# Single Runtime Master Sequence - Recurrent Qwen

Use this notebook inside one Colab runtime. It fetches the current bootstrap from GitHub by resolved commit SHA and executes one explicit `STAGE5_CURRENT_A100_TARGET` at a time.

Current GPU queue: `reentry_repair_smoke` -> review/status -> `reentry_recovery_training` -> `debiased_benchmark_suite` -> `dense_mcq_trace_sft_control`. Parallel CPU/API data prep is exposed separately as `claim_curriculum_scaleup_cpu` and should not be treated as a GPU gate.

Run the setup cell once, then run the next target cell. Keep `KEEP_RUNTIME_OPEN = False` unless you intentionally want to run multiple cells in one attached runtime; the default conserves credits by letting target cells disconnect when they finish.


In [ ]:
import base64, json, os, time, urllib.request
from google.colab import userdata

REPO = "mshapiro123/recurrent-qwen-svgd"
KEEP_RUNTIME_OPEN = False

DISCONNECT_ENV_VARS = {
    "STAGE5_MASTER_SEQUENCE_STATUS_DISCONNECT",
    "STAGE5_REENTRY_REPAIR_DISCONNECT",
    "STAGE5_REENTRY_RECOVERY_DISCONNECT",
    "STAGE5_DEBIASED_BENCHMARK_DISCONNECT",
    "STAGE5_DENSE_MCQ_DISCONNECT",
    "STAGE5_CURRICULUM_PIPELINE_DISCONNECT",
}


def _secret(*names):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            return value
    return None


def _gh_json(url, gh_token):
    req = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {gh_token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.load(response)


def launch_stage5_target(target, *, keep_runtime_open=KEEP_RUNTIME_OPEN, extra_env=None):
    gh = _secret("GH_TOKEN", "GITHUB_TOKEN")
    assert gh, "Missing GH_TOKEN or GITHUB_TOKEN in Colab secrets."

    hf = _secret("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
        os.environ["HUGGINGFACE_HUB_TOKEN"] = hf
        print("HF token loaded from Colab secrets.")
    else:
        print("WARNING: HF token not found; downloads may use anonymous Hub access.")

    if keep_runtime_open:
        for name in DISCONNECT_ENV_VARS:
            os.environ[name] = "0"
    if extra_env:
        for key, value in extra_env.items():
            os.environ[key] = str(value)

    os.environ["STAGE5_CURRENT_A100_TARGET"] = target

    resolved_ref = _gh_json(
        f"https://api.github.com/repos/{REPO}/git/refs/heads/main?cache_bust={time.time_ns()}",
        gh,
    )["object"]["sha"]

    payload = _gh_json(
        f"https://api.github.com/repos/{REPO}/contents/colab/CURRENT_A100_BOOTSTRAP_CELL.py"
        f"?ref={resolved_ref}&cache_bust={time.time_ns()}",
        gh,
    )

    code_blob = base64.b64decode(payload["content"]).decode("utf-8")
    required = [
        "sha_resolved_nested_fetch_v3",
        target,
        "STAGE5_CURRENT_A100_TARGET",
    ]
    missing = [marker for marker in required if marker not in code_blob]
    assert not missing, f"Fetched stale or incomplete bootstrap: {missing}"
    print("Fetched bootstrap sha:", payload.get("sha"), "commit:", resolved_ref[:12], "target:", target)
    exec(compile(code_blob, "colab/CURRENT_A100_BOOTSTRAP_CELL.py", "exec"))


## 0. Cheap Status / Review

Use after a runtime restart or after a target publishes. This does not download models or train; it prints the current pointer and reviewer recommendation.


In [ ]:
launch_stage5_target("master_sequence_status")


## 1. Stage 3 - Trainable Re-entry Repair Smoke

Run on L4/T4. This is the current front-of-queue target. Continue only if the reviewer recommends bounded recovery training.


In [ ]:
launch_stage5_target("reentry_repair_smoke")


## 2. Stage 4 - Bounded Deterministic Recovery SFT

Run only after Stage 3 passes. This trains deterministic depth recovery with the repaired loop-closure path and publishes a wrapper summary.


In [ ]:
launch_stage5_target("reentry_recovery_training")


## 3. Phase 1 Benchmark - Base vs Repaired Recurrent

Run only after Stage 4 validation is sane. This evaluates base Qwen 0.5B versus the repaired recurrent checkpoint using debiased MCQ scoring. After it publishes, run `master_sequence_status` and follow the `Phase 1 Gate Review` section before launching the dense control.


In [ ]:
launch_stage5_target("debiased_benchmark_suite")


## 4. Same-Curriculum Dense Control

Run only after the debiased benchmark passes and the `Phase 1 Gate Review` asks for `dense_mcq_trace_sft_control`. This trains/evaluates standard dense Qwen LoRA on the same curriculum so the architecture claim is not confused with data-only improvement. After it publishes, run `master_sequence_status` again; do not advance to Phase 2 until the Phase 1 gate reports an architecture signal.


In [ ]:
launch_stage5_target("dense_mcq_trace_sft_control")


## 5. Parallel CPU/API Curriculum Scale-Up

Run this on CPU or a cheap non-GPU runtime while the paid GPU sequence stays on Phase 0/1. It prepares the claim-sized direct/deep curriculum shard; it does not replace the re-entry repair and recovery gates. Keep provider calls disabled until model ids and API secrets are configured. For a tiny paid provider smoke, pass `extra_env={"STAGE5_CURRICULUM_RUN_PROVIDER_RESPONSES": "1", "STAGE5_CURRICULUM_PROVIDER_LIMIT": "2"}`.


In [ ]:
launch_stage5_target("claim_curriculum_scaleup_cpu")


## Downstream Gates

Do not run Phase 2 breadth diagnostics until the recurrent checkpoint is competitive with base, has been compared against the dense same-curriculum control, and the `Phase 1 Gate Review` reports a real architecture signal. Do not resume particles/SVGD until breadth is correct-bearing under the Phase 2 diagnostic.
